# 1. LEITURA DAS TABELAS DA CAMADA SILVER

A camada Gold é responsável pela aplicação das regras de negócio e pela disponibilização dos dados em formatos adequados para análise.

Nesta etapa são carregadas as tabelas produzidas na camada Silver, que servirão como base para a construção dos indicadores e simulações do projeto.

As tabelas geradas nesta camada serão utilizadas posteriormente na análise exploratória e na resposta às perguntas de negócio definidas para o MVP.

In [0]:
# importações

from pyspark.sql.functions import (
    avg,
    min,
    max,
    stddev,
    lit,
    round,
    col,
    to_date,
    regexp_replace
    )

In [0]:
silver_indicadores_macro = spark.table(
    "silver_indicadores_macro"
)

silver_quality_report = spark.table(
    "silver_quality_report"
)

dim_calendario = spark.table(
    "dim_calendario"
)

In [0]:
display(silver_indicadores_macro)

ano_mes,cdi_percentual,selic_percentual,ipca_percentual
2023-12,11.65,11.75,0.56
2024-01,11.65,11.75,0.42
2024-02,11.15,11.25,0.83
2024-03,10.65,10.75,0.16
2024-04,10.65,10.75,0.38
2024-05,10.4,10.5,0.46
2024-06,10.4,10.5,0.21
2024-07,10.4,10.5,0.38
2024-08,10.4,10.5,-0.02
2024-09,10.65,10.75,0.44


In [0]:
display(silver_quality_report)

quantidade_registros,data_inicio,data_fim,indicador
120,2016-07-31,2026-06-30,Selic
120,2016-07-31,2026-06-30,Ipca
120,2016-07-29,2026-06-30,CDI


In [0]:
display(dim_calendario)

data_referencia,ano,mes,trimestre,semestre,nome_mes,ano_mes
2023-03-21,2023,3,1,1,March,2023-3
2023-05-04,2023,5,2,1,May,2023-5
2023-05-05,2023,5,2,1,May,2023-5
2023-05-29,2023,5,2,1,May,2023-5
2023-06-22,2023,6,2,1,June,2023-6
2023-08-01,2023,8,3,2,August,2023-8
2023-08-18,2023,8,3,2,August,2023-8
2023-10-04,2023,10,4,2,October,2023-10
2023-10-09,2023,10,4,2,October,2023-10
2023-10-26,2023,10,4,2,October,2023-10


# 2. CONSTRUÇÃO DA TABELA GOLD_EVOLUCAO_INDICADORES

Esta tabela tem como objetivo disponibilizar uma visão consolidada da evolução dos indicadores econômicos utilizados no estudo.

A estrutura resultante servirá como base para análises temporais, comparações entre os indicadores e construção de visualizações analíticas na etapa de exploração dos dados.

In [0]:
gold_evolucao_indicadores = (
    silver_indicadores_macro
    .orderBy("ano_mes")
)

In [0]:
display(gold_evolucao_indicadores)

ano_mes,cdi_percentual,selic_percentual,ipca_percentual
2016-07,14.13,14.25,0.52
2016-08,14.13,14.25,0.44
2016-09,14.13,14.25,0.08
2016-10,13.88,14.0,0.26
2016-11,13.88,14.0,0.18
2016-12,13.63,13.75,0.3
2017-01,12.88,13.0,0.38
2017-02,12.13,12.25,0.33
2017-03,12.13,12.25,0.25
2017-04,11.13,11.25,0.14


In [0]:
gold_evolucao_indicadores.write \
    .mode("overwrite") \
    .saveAsTable("gold_evolucao_indicadores")

# 3. CONSTRUÇÃO DA TABELA GOLD_RISCO_INDEXADORES

Esta tabela tem como objetivo consolidar métricas estatísticas dos indicadores econômicos analisados no projeto.

Os indicadores calculados permitem avaliar o comportamento histórico de cada série e apoiar a análise de volatilidade dos indexadores.

As métricas consideradas são:

- Média;
- Valor mínimo;
- Valor máximo;
- Desvio padrão.

Esses indicadores serão utilizados posteriormente para identificar qual série apresentou maior variabilidade ao longo do período analisado.

In [0]:
risco_cdi = (
    silver_indicadores_macro
    .agg(
        avg("cdi_percentual").alias("media"),
        min("cdi_percentual").alias("minimo"),
        max("cdi_percentual").alias("maximo"),
        stddev("cdi_percentual").alias("desvio_padrao")
    )
    .withColumn("indicador", lit("CDI"))
)

In [0]:
risco_selic = (
    silver_indicadores_macro
    .agg(
        avg("selic_percentual").alias("media"),
        min("selic_percentual").alias("minimo"),
        max("selic_percentual").alias("maximo"),
        stddev("selic_percentual").alias("desvio_padrao")
    )
    .withColumn("indicador", lit("Selic"))
)

In [0]:
risco_ipca = (
    silver_indicadores_macro
    .agg(
        avg("ipca_percentual").alias("media"),
        min("ipca_percentual").alias("minimo"),
        max("ipca_percentual").alias("maximo"),
        stddev("ipca_percentual").alias("desvio_padrao")
    )
    .withColumn("indicador", lit("IPCA"))
)

In [0]:
gold_risco_indexadores = (
    risco_cdi
    .unionByName(risco_selic)
    .unionByName(risco_ipca)
)

In [0]:
display(gold_risco_indexadores)


media,minimo,maximo,desvio_padrao,indicador
9.480166666666664,1.9,14.9,4.229930166762361,CDI
9.583333333333334,2.0,15.0,4.230781228558334,Selic
0.40925,-0.68,1.62,0.3820360370612247,IPCA


In [0]:
gold_risco_indexadores.write \
    .mode("overwrite") \
    .saveAsTable("gold_risco_indexadores")

In [0]:
gold_risco_indexadores = (
    gold_risco_indexadores
    .withColumn("media", round("media", 4))
    .withColumn("minimo", round("minimo", 4))
    .withColumn("maximo", round("maximo", 4))
    .withColumn("desvio_padrao", round("desvio_padrao", 4))
)

In [0]:
gold_risco_indexadores.write \
    .mode("overwrite") \
    .saveAsTable("gold_risco_indexadores")

In [0]:
display(gold_risco_indexadores)

media,minimo,maximo,desvio_padrao,indicador
9.4802,1.9,14.9,4.2299,CDI
9.5833,2.0,15.0,4.2308,Selic
0.4093,-0.68,1.62,0.382,IPCA


# 4. LIMITAÇÕES IDENTIFICADAS NA MODELAGEM DE FINANCIAMENTO

Durante o desenvolvimento da camada Gold foi avaliada a possibilidade de simular a evolução de um financiamento indexado ao CDI e compará-lo ao IPCA.

A série SGS 4389 - CDI, foi selecionada por sua aderência aos objetivos de análise temporal, comparação entre indicadores econômicos e avaliação de risco, atendendo adequadamente às necessidades analíticas propostas para este estudo. No entanto, por se tratar de uma taxa anualizada base 252 dias úteis, sua utilização direta na modelagem de uma taxa mensal efetiva para simulação de financiamentos demandaria tratamentos adicionais e séries complementares específicas para esse fim. 

Considerando a necessidade de preservar a consistência metodológica dos resultados, optou-se por não realizar uma simulação financeira baseada em premissas que poderiam comprometer a interpretação dos dados.



In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+--------+-------------------------+-----------+
|database|tableName                |isTemporary|
+--------+-------------------------+-----------+
|default |bronze_cdi               |false      |
|default |bronze_ipca              |false      |
|default |bronze_selic             |false      |
|default |dim_calendario           |false      |
|default |gold_evolucao_indicadores|false      |
|default |gold_risco_indexadores   |false      |
|default |silver_indicadores_macro |false      |
|default |silver_quality_report    |false      |
+--------+-------------------------+-----------+



# Conclusão da Camada Gold

A camada Gold foi responsável por transformar os dados consolidados da camada Silver em estruturas orientadas ao negócio.

Como resultado, foram construídas as seguintes tabelas analíticas:

- `gold_evolucao_indicadores`, utilizada para análise da evolução histórica dos indicadores econômicos;
- `gold_risco_indexadores`, utilizada para avaliação do comportamento estatístico e da volatilidade dos indicadores analisados.

Durante o desenvolvimento desta camada também foi realizada uma avaliação da possibilidade de simulação de financiamentos indexados ao CDI e ao IPCA. Entretanto, devido às características metodológicas da série CDI utilizada (SGS 4389), optou-se por não construir essa simulação, preservando a consistência analítica dos resultados apresentados.

As tabelas produzidas nesta etapa servirão como fonte para a construção das análises, visualizações e conclusões do projeto.
